
# Laboratory 5A — Understand the multi-volume QHA dataset

The expensive calculations have already been completed. Your first task is not
to run Quantas and not to read a summary table: it is to understand what the
supplied CRYSTAL files actually contain.

Scientific analysis begins by checking the provenance of the data that enter
the model.


In [ ]:

from pathlib import Path

raw = Path("reference-results/raw")
inputs = sorted(raw.glob("*.d12"))
outputs = sorted(raw.glob("*.out"))

print(f"Input files : {len(inputs)}")
print(f"Output files: {len(outputs)}")
for path in inputs:
    print(path.name)



## 1. Inspect a calculation manually

Open the central-volume files as plain text:

```text
reference-results/raw/mgo_v100_phonons.d12
reference-results/raw/mgo_v100_phonons.out
```

In the input, search for:

```text
SCELPHONO
BASISSET
DFT
SHRINK
```

In the output, search for phrases containing:

```text
NUMBER OF ATOMS PER SUPERCELL
PERMITS THE CALCULATION OF MODES AT
NUMBER OF AO
NUMBER OF SHELLS
```

Do not use the automated summary below until you have found these entries
yourself.



### Question 1 — What calculation was performed?

From the **input file**, record:

- the `SCELPHONO` matrix;
- the exchange–correlation functional;
- the basis set;
- the electronic k-point sampling (`SHRINK`).

From the **output file**, record:

- the number of atoms in the phonon supercell;
- the number of phonon q-points;
- the number of atomic orbitals.

**Your answer:**  
_Write the values here, together with the keywords/sections where you found them._



### Question 2 — What does the supercell imply?

Using the `SCELPHONO` matrix that you found, calculate the number of primitive
cells represented by the phonon supercell.

MgO has two atoms in its primitive cell. Is your result consistent with the
number of atoms printed by CRYSTAL?

**Your answer:**  
_Show the short calculation._



## 2. Check your manual inspection

Now use the course parser to summarize all seven calculations.

This table is a **check**, not a substitute for reading the files. If one of
your manually recorded values disagrees with the table, return to the raw
input/output and resolve the discrepancy.


In [ ]:

from mineral_science_tools.qha_lab import reference_series_table

series = reference_series_table("reference-results/raw")
series



### Question 3 — Consistency across the volume series

Which settings remain identical at all seven volumes?

Why is this consistency necessary before interpreting changes in the phonon
spectrum as a response to volume rather than as a change in numerical setup?

**Your answer:**  
_Write 3–4 lines here._



## 3. Inspect the sampled volume interval


In [ ]:

series[[
    "V/Vref label",
    "a conventional (A)",
    "V conventional (A^3)",
    "V/Vref from cell",
]]


In [ ]:

import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.scatter(series["V/Vref label"], series["V conventional (A^3)"])
ax.axvline(1.0, linewidth=0.8, linestyle="--")
ax.set_xlabel("V / Vref")
ax.set_ylabel("Conventional cell volume (Å³)")
fig.tight_layout()



### Question 4 — Is the volume range physically adequate?

Imagine constructing the same QHA analysis using only points with
$V/V_\mathrm{ref}\leq1$.

Would that dataset be sufficient for a finite-temperature calculation at
ambient pressure?

Use the equilibrium condition discussed in the course material to justify your
answer.

**Your answer:**  
_Write 4–5 lines here._



## 4. A computational-cost check

Return to the output and look at the number of atoms, atomic orbitals and
shells.

### Question 5

Why is it reasonable to provide these phonon calculations as reference data
rather than asking every student to recompute all seven volumes during the
laboratory?

Your answer should refer to the calculation size and to what must be repeated
in a numerical phonon calculation.

**Your answer:**  
_Write 3–4 lines here._



## 5. Build the Quantas QHA input

Before running the generator, validate the distributed list of CRYSTAL outputs.
The paths in the list are interpreted from the laboratory working directory.


In [ ]:

from mineral_science_tools.qha_lab import validate_reference_list

validate_reference_list("reference-results/files.txt")


In [ ]:

print(Path("reference-results/files.txt").read_text())



The files are ordered from the most compressed to the most expanded structure.
The central $V/V_\mathrm{ref}=1.00$ calculation is the fourth entry, so its
Python-style reference index is `3`.


In [ ]:

!quantas qha inpgen reference-results/files.txt \
    --list \
    --interface crystal \
    --reference 3 \
    --output mgo_qha.yaml \
    --jobname "MgO multi-volume QHA teaching dataset"



### Question 6 — Why does branch correspondence matter?

Read the generator message and inspect the beginning of `mgo_qha.yaml`.

Why must the mode labelled $(\mathbf q,j)$ at one volume correspond to the
same physical branch at the other volumes if the frequency scheme interpolates

$$
\nu_{\mathbf qj}(V)?
$$

**Your answer:**  
_Write 3–4 lines here._



## 6. Inspect the pressure support


In [ ]:

!quantas qha inspect mgo_qha.yaml \
    --degree 3 \
    --eos BM3 \
    --report qha_inspect.log



### Question 7 — Does the input support the intended pressure range?

Use the `qha inspect` output to decide whether the sampled volumes bracket the
pressure interval that will be explored later.

Compare the polynomial and BM3 estimates, and identify whether any conclusion
would rely strongly on extrapolation.

**Your answer:**  
_Write 4–5 lines here._
